# C2.6 · Building the research harness

**Function C — Offensive Security & Research → The Security Researcher**  ·  *AI for Security*

---

**Risk.** Model effects and harness effects confounded.

**Control.** Multi-backbone benchmarking with statistical honesty.

**This lab.** Separate model effects from harness effects.

| | |
|---|---|
| Open-source tooling | Inspect, Cyber Commons eval harness |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C2.6"))

The research harness is the difference between a person who finds things and a capability that keeps finding them.

In [ ]:
from cybercommons import research, redteam

# a harness = a suite + a target adapter + a recorded rate
def sweep(technique_rate, n=200, seed=3):
    return research.trial(lambda rng: rng.random() < technique_rate, n=n, seed=seed)

BASELINE = {"INJ-01": 0.05, "INJ-02": 0.40, "INJ-03": 0.55, "INJ-04": 0.70}
print(f"{'attack':8s}{'rate':>8}{'ci95':>18}  verdict")
for aid, p in BASELINE.items():
    r = sweep(p)
    print(f"{aid:8s}{r['rate']:>8.3f}{str(r['ci95']):>18}  {r['verdict']}")

Now the property that makes it a harness rather than a script: the same suite re-runs after a change and the deltas are comparable.

In [ ]:
MITIGATED = {"INJ-01": 0.00, "INJ-02": 0.02, "INJ-03": 0.03, "INJ-04": 0.05}
print(f"{'attack':8s}{'before':>9}{'after':>9}{'delta':>9}")
for aid in BASELINE:
    b, a = sweep(BASELINE[aid])["rate"], sweep(MITIGATED[aid])["rate"]
    print(f"{aid:8s}{b:>9.3f}{a:>9.3f}{a - b:>+9.3f}")
print("\ncoverage:", redteam.coverage())

### Expect

Four baseline rates print with intervals and verdicts, then a before/after table showing large negative deltas, plus the suite's surface coverage.

### Your turn

The harness above measures only the injection surface. Extend the sweep to identity and containment, then state honestly which surface you have the least evidence about.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C2.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*